In [4]:
!pip install polars

   ---------------------------------------- 0.0/828.7 kB ? eta -:--:--
   ---------------------------------------- 10.2/828.7 kB ? eta -:--:--
   - ------------------------------------- 41.0/828.7 kB 653.6 kB/s eta 0:00:02
   ---------------- ----------------------- 337.9/828.7 kB 3.5 MB/s eta 0:00:01
   ---------------------------------------  819.2/828.7 kB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 828.7/828.7 kB 4.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/51.8 MB ? eta -:--:--
   - -------------------------------------- 1.9/51.8 MB 59.5 MB/s eta 0:00:01
   - -------------------------------------- 2.5/51.8 MB 53.8 MB/s eta 0:00:01
   ---- ----------------------------------- 5.5/51.8 MB 43.6 MB/s eta 0:00:02
   ----- ---------------------------------- 6.9/51.8 MB 44.3 MB/s eta 0:00:02
   ------ --------------------------------- 8.7/51.8 MB 39.7 MB/s eta 0:00:02
   ------- -------------------------------- 10.4/51.8 MB 40.9 MB/s eta 0:00:0

In [5]:
import os, time
import numpy as np
import polars as pl
from sklearn.neighbors import BallTree

OUTPUTS_DIR = "outputs"
STATIONS_PATH = "old_aadt.csv"

EARTH_RADIUS_MILES = 3958.8
MAX_STATION_MILES = 1.0
TX_LAT_MIN, TX_LAT_MAX = 25.8, 36.5
TX_LON_MIN, TX_LON_MAX = -106.6, -93.5
SAMPLE_SEED = 42

REQUIRED_COLS = [
    "Crash_ID", "Latitude", "Longitude", "ZIP_Code",
    "Adt_Curnt_Amt", "Distance_Miles", "Station_Year",
    "year_gap", "aadt_match_type", "VMT_Multiplier",
]

TX_VMT_MILLIONS = {
    2015: 258300, 2016: 270700, 2017: 273200, 2018: 282200, 2019: 288400,
    2020: 260000, 2021: 285200, 2022: 291100, 2023: 301500, 2024: 307800,
    2025: 307800,
}

ATOL_VMT = 1e-9
ATOL_DIST = 1e-4

results = []

def record(name, passed, msg):
    results.append({"validation": name, "passed": bool(passed), "msg": msg})
    status = "pass" if passed else "FAIL"
    print(f"[{status}] {name}: {msg}")


In [6]:
# ── Discover yearly output files ─────────────────────────────────────────────

year_files = []

for f in sorted(os.listdir(OUTPUTS_DIR)):
    if f.startswith("Cleaned_Data_") and f.endswith(".csv") and "_Final_" not in f:
        try:
            y = int(f.replace("Cleaned_Data_", "").replace(".csv", ""))
            year_files.append((y, os.path.join(OUTPUTS_DIR, f)))
        except ValueError:
            pass

year_files.sort()

print(f"{len(year_files)} yearly files:")
for y, p in year_files:
    print(f"  {y}: {p} ({os.path.getsize(p) / 1e6:.0f} MB)")

9 yearly files:
  2016: outputs\Cleaned_Data_2016.csv (198 MB)
  2017: outputs\Cleaned_Data_2017.csv (194 MB)
  2018: outputs\Cleaned_Data_2018.csv (196 MB)
  2019: outputs\Cleaned_Data_2019.csv (200 MB)
  2020: outputs\Cleaned_Data_2020.csv (169 MB)
  2021: outputs\Cleaned_Data_2021.csv (196 MB)
  2022: outputs\Cleaned_Data_2022.csv (195 MB)
  2023: outputs\Cleaned_Data_2023.csv (149 MB)
  2024: outputs\Cleaned_Data_2024.csv (197 MB)


In [7]:
# ── V01 Schema validation ───────────────────────────────────────────────────

for year, path in year_files:
    header = pl.read_csv(path, n_rows=0).columns
    missing = [c for c in REQUIRED_COLS if c not in header]

    record(
        f"V01 schema {year}",
        len(missing) == 0,
        f"missing {missing}" if missing else f"{len(REQUIRED_COLS)}/10 present, {len(header)} cols total"
    )

[pass] V01 schema 2016: 10/10 present, 92 cols total
[pass] V01 schema 2017: 10/10 present, 92 cols total
[pass] V01 schema 2018: 10/10 present, 92 cols total
[pass] V01 schema 2019: 10/10 present, 92 cols total
[pass] V01 schema 2020: 10/10 present, 92 cols total
[pass] V01 schema 2021: 10/10 present, 92 cols total
[pass] V01 schema 2022: 10/10 present, 92 cols total
[pass] V01 schema 2023: 10/10 present, 92 cols total
[pass] V01 schema 2024: 10/10 present, 92 cols total


In [8]:
# ── V02 Crash year validation ───────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Crash_Year"])
    yrs = df.select(pl.col("Crash_Year").drop_nulls().unique()).to_series().to_list()

    ok = len(yrs) == 1 and int(yrs[0]) == year

    record(
        f"V02 Crash_Year {year}",
        ok,
        f"distinct values = {sorted(yrs)}"
    )


[pass] V02 Crash_Year 2016: distinct values = [2016]
[pass] V02 Crash_Year 2017: distinct values = [2017]
[pass] V02 Crash_Year 2018: distinct values = [2018]
[pass] V02 Crash_Year 2019: distinct values = [2019]
[pass] V02 Crash_Year 2020: distinct values = [2020]
[pass] V02 Crash_Year 2021: distinct values = [2021]
[pass] V02 Crash_Year 2022: distinct values = [2022]
[pass] V02 Crash_Year 2023: distinct values = [2023]
[pass] V02 Crash_Year 2024: distinct values = [2024]


In [9]:
# ── V03 Coordinate bounds validation ────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Latitude", "Longitude"])

    out = df.select([
        (
            pl.col("Latitude").is_null() |
            pl.col("Longitude").is_null()
        ).sum().alias("null_count"),

        (
            (pl.col("Latitude") < TX_LAT_MIN) |
            (pl.col("Latitude") > TX_LAT_MAX) |
            (pl.col("Longitude") < TX_LON_MIN) |
            (pl.col("Longitude") > TX_LON_MAX)
        ).sum().alias("out_of_bounds")
    ])

    null_count = out["null_count"][0]
    out_of_bounds = out["out_of_bounds"][0]

    record(
        f"V03 bounds {year}",
        null_count == 0 and out_of_bounds == 0,
        f"{null_count} null, {out_of_bounds} out-of-TX"
    )


[pass] V03 bounds 2016: 0 null, 0 out-of-TX
[pass] V03 bounds 2017: 0 null, 0 out-of-TX
[pass] V03 bounds 2018: 0 null, 0 out-of-TX
[pass] V03 bounds 2019: 0 null, 0 out-of-TX
[pass] V03 bounds 2020: 0 null, 0 out-of-TX
[pass] V03 bounds 2021: 0 null, 0 out-of-TX
[pass] V03 bounds 2022: 0 null, 0 out-of-TX
[pass] V03 bounds 2023: 0 null, 0 out-of-TX
[pass] V03 bounds 2024: 0 null, 0 out-of-TX


In [10]:
# ── V04 Distance cap validation ─────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Distance_Miles", "Adt_Curnt_Amt"])

    out = df.select([
        (
            (pl.col("Distance_Miles") > MAX_STATION_MILES) &
            pl.col("Adt_Curnt_Amt").is_not_null()
        ).sum().alias("violations"),

        pl.col("Distance_Miles")
        .filter(pl.col("Adt_Curnt_Amt").is_not_null())
        .max()
        .alias("max_filled")
    ])

    violations = out["violations"][0]
    max_filled = out["max_filled"][0]

    record(
        f"V04 dist cap {year}",
        violations == 0,
        f"{violations} violations; max Distance_Miles among filled = {max_filled:.4f}"
    )


[pass] V04 dist cap 2016: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2017: 0 violations; max Distance_Miles among filled = 0.9999
[pass] V04 dist cap 2018: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2019: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2020: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2021: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2022: 0 violations; max Distance_Miles among filled = 1.0000
[pass] V04 dist cap 2023: 0 violations; max Distance_Miles among filled = 0.9999
[pass] V04 dist cap 2024: 0 violations; max Distance_Miles among filled = 1.0000


In [11]:
# ── V05 ZIP validation ──────────────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["ZIP_Code"])

    nulls = df.select(pl.col("ZIP_Code").is_null().sum()).item()

    record(
        f"V05 ZIP {year}",
        nulls == 0,
        f"{nulls} null ZIP rows"
    )


[pass] V05 ZIP 2016: 0 null ZIP rows
[pass] V05 ZIP 2017: 0 null ZIP rows
[pass] V05 ZIP 2018: 0 null ZIP rows
[pass] V05 ZIP 2019: 0 null ZIP rows
[pass] V05 ZIP 2020: 0 null ZIP rows
[pass] V05 ZIP 2021: 0 null ZIP rows
[pass] V05 ZIP 2022: 0 null ZIP rows
[pass] V05 ZIP 2023: 0 null ZIP rows
[pass] V05 ZIP 2024: 0 null ZIP rows


In [12]:
# ── V06 Year gap validation ─────────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Crash_Year", "Station_Year", "year_gap"])

    df = df.drop_nulls(["Station_Year", "year_gap"])

    mismatches = df.select(
        (
            (
                (pl.col("Crash_Year").cast(pl.Int64) - pl.col("Station_Year").cast(pl.Int64)).abs()
                != pl.col("year_gap").cast(pl.Int64)
            )
        ).sum()
    ).item()

    record(
        f"V06 year_gap {year}",
        mismatches == 0,
        f"{mismatches} mismatches"
    )

[pass] V06 year_gap 2016: 0 mismatches
[pass] V06 year_gap 2017: 0 mismatches
[pass] V06 year_gap 2018: 0 mismatches
[pass] V06 year_gap 2019: 0 mismatches
[pass] V06 year_gap 2020: 0 mismatches
[pass] V06 year_gap 2021: 0 mismatches
[pass] V06 year_gap 2022: 0 mismatches
[pass] V06 year_gap 2023: 0 mismatches
[pass] V06 year_gap 2024: 0 mismatches


In [13]:
# ── V07 VMT multiplier validation ───────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Crash_Year", "Station_Year", "VMT_Multiplier"])

    df = df.drop_nulls(["VMT_Multiplier", "Station_Year"])

    df = df.with_columns([
        pl.col("Crash_Year").cast(pl.Int64).replace(TX_VMT_MILLIONS).alias("VMT_Crash"),
        pl.col("Station_Year").cast(pl.Int64).replace(TX_VMT_MILLIONS).alias("VMT_Station"),
    ])

    df = df.with_columns(
        (pl.col("VMT_Crash") / pl.col("VMT_Station")).alias("expected")
    )

    mismatches = df.select(
        (
            ((pl.col("expected") - pl.col("VMT_Multiplier")).abs() / pl.col("expected").abs())
            > ATOL_VMT
        ).sum()
    ).item()

    record(
        f"V07 VMT {year}",
        mismatches == 0,
        f"{mismatches} mismatches (rtol > {ATOL_VMT:.0e})"
    )


[pass] V07 VMT 2016: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2017: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2018: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2019: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2020: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2021: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2022: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2023: 0 mismatches (rtol > 1e-09)
[pass] V07 VMT 2024: 0 mismatches (rtol > 1e-09)


In [14]:
# ── V08 AADT calculation validation ─────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Station_Count", "VMT_Multiplier", "Adt_Curnt_Amt"])

    df = df.drop_nulls(["Adt_Curnt_Amt"])

    df = df.with_columns(
        (pl.col("Station_Count") * pl.col("VMT_Multiplier")).alias("expected")
    )

    mismatches = df.select(
        (
            ((pl.col("expected") - pl.col("Adt_Curnt_Amt")).abs() / pl.col("Adt_Curnt_Amt").abs())
            > 1e-6
        ).sum()
    ).item()

    record(
        f"V08 AADT {year}",
        mismatches == 0,
        f"{mismatches} mismatches (rtol > 1e-6)"
    )


[pass] V08 AADT 2016: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2017: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2018: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2019: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2020: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2021: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2022: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2023: 0 mismatches (rtol > 1e-6)
[pass] V08 AADT 2024: 0 mismatches (rtol > 1e-6)


In [15]:
# ── V09 Match type validation ───────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Adt_Curnt_Amt", "aadt_match_type"])

    out = df.select([
        (
            pl.col("Adt_Curnt_Amt").is_not_null() &
            (pl.col("aadt_match_type") != "NEAREST_STATION_VMT_NORM")
        ).sum().alias("bad_filled"),

        (
            pl.col("Adt_Curnt_Amt").is_null() &
            (pl.col("aadt_match_type") != "MISSING")
        ).sum().alias("bad_missing")
    ])

    bad_filled = out["bad_filled"][0]
    bad_missing = out["bad_missing"][0]

    record(
        f"V09 match_type {year}",
        bad_filled == 0 and bad_missing == 0,
        f"{bad_filled} filled+wrong-label, {bad_missing} missing+wrong-label"
    )


[pass] V09 match_type 2016: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2017: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2018: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2019: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2020: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2021: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2022: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2023: 0 filled+wrong-label, 0 missing+wrong-label
[pass] V09 match_type 2024: 0 filled+wrong-label, 0 missing+wrong-label


In [16]:
# ── Load station data for BallTree validation ───────────────────────────────

stations = pl.read_csv(STATIONS_PATH, infer_schema_length=10000)
stations = stations.rename({c: c.strip() for c in stations.columns})

LAT_COL = "LATITUDE"
LON_COL = "LONGITUDE"
LOC_COL = "TRFC_STATN_ID"
COUNT_COL = "LATEST_AADT_QTY"
YEAR_COL = "LATEST_AADT_YR"

stations = stations.with_columns([
    pl.col(LAT_COL).cast(pl.Float64, strict=False),
    pl.col(LON_COL).cast(pl.Float64, strict=False),
    pl.col(COUNT_COL).cast(pl.Float64, strict=False),
    pl.col(YEAR_COL).cast(pl.Int64, strict=False),
    pl.col(LOC_COL).cast(pl.Utf8).str.strip_chars(),
])

stations = (
    stations
    .drop_nulls([LAT_COL, LON_COL, COUNT_COL])
    .filter(pl.col(COUNT_COL) > 0)
    .sort(YEAR_COL)
    .group_by(LOC_COL, maintain_order=True)
    .tail(1)
)

print(f"Stations after cleaning: {stations.height:,}")

station_pd = stations.select([LAT_COL, LON_COL, LOC_COL]).to_pandas()

stations_coords_rad = np.radians(station_pd[[LAT_COL, LON_COL]].values)
tree = BallTree(stations_coords_rad, metric="haversine")

stat_lat_arr = station_pd[LAT_COL].values
stat_lon_arr = station_pd[LON_COL].values
stat_id_arr = station_pd[LOC_COL].values


def brute_force_nearest(crash_lat, crash_lon):
    lat1 = np.radians(crash_lat)
    lon1 = np.radians(crash_lon)
    lat2 = np.radians(stat_lat_arr)
    lon2 = np.radians(stat_lon_arr)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2 +
        np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    d = EARTH_RADIUS_MILES * c

    idx = int(np.argmin(d))

    return stat_id_arr[idx], float(d[idx])


Stations after cleaning: 100,400


In [17]:
# ── V10 BallTree vs brute-force sample check ────────────────────────────────

target_year, target_path = year_files[-1]

df_recent = pl.read_csv(
    target_path,
    columns=["Latitude", "Longitude", "Nearest_Station_LocationId", "Distance_Miles"]
)

df_recent = df_recent.drop_nulls(["Latitude", "Longitude", "Nearest_Station_LocationId"])

sample_n = min(1000, df_recent.height)

sample = (
    df_recent
    .sample(n=sample_n, seed=SAMPLE_SEED)
    .to_pandas()
    .reset_index(drop=True)
)

id_mismatch = 0
dist_mismatch = 0
worst_dist_err = 0.0
example_bad = []

for _, r in sample.iterrows():
    true_id, true_dist = brute_force_nearest(r["Latitude"], r["Longitude"])
    stored_id = str(r["Nearest_Station_LocationId"]).strip()

    if str(true_id) != stored_id:
        id_mismatch += 1

        if len(example_bad) < 5:
            example_bad.append((stored_id, str(true_id), r["Distance_Miles"], true_dist))

    dist_err = abs(true_dist - float(r["Distance_Miles"]))
    worst_dist_err = max(worst_dist_err, dist_err)

    if dist_err > ATOL_DIST:
        dist_mismatch += 1

ok = id_mismatch == 0 and dist_mismatch == 0

record(
    f"V10 BallTree vs brute-force (n={len(sample)}, year={target_year})",
    ok,
    f"{id_mismatch} ID mismatches, {dist_mismatch} distance mismatches (max diff {worst_dist_err:.2e} mi)"
)

if not ok:
    print("First mismatches (stored_id, brute_id, stored_dist, brute_dist):")
    for row in example_bad:
        print(f"  {row}")

[pass] V10 BallTree vs brute-force (n=1000, year=2024): 0 ID mismatches, 0 distance mismatches (max diff 1.11e-16 mi)


In [18]:
# ── V11 Speed comparison ────────────────────────────────────────────────────

bench_n = min(10_000, df_recent.height)

bench = (
    df_recent
    .sample(n=bench_n, seed=SAMPLE_SEED)
    .select(["Latitude", "Longitude"])
    .to_numpy()
)

t0 = time.perf_counter()
crash_rad = np.radians(bench)
distances_rad, indices = tree.query(crash_rad, k=1)
t_balltree = time.perf_counter() - t0

t0 = time.perf_counter()

chunk = 500
brute_indices = np.empty(bench_n, dtype=np.int64)

stat_lat_r = np.radians(stat_lat_arr)
stat_lon_r = np.radians(stat_lon_arr)

for start in range(0, bench_n, chunk):
    end = min(start + chunk, bench_n)

    lat1 = np.radians(bench[start:end, 0])[:, None]
    lon1 = np.radians(bench[start:end, 1])[:, None]

    dlat = stat_lat_r[None, :] - lat1
    dlon = stat_lon_r[None, :] - lon1

    a = (
        np.sin(dlat / 2) ** 2 +
        np.cos(lat1) * np.cos(stat_lat_r[None, :]) *
        np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    d = EARTH_RADIUS_MILES * c

    brute_indices[start:end] = np.argmin(d, axis=1)

t_brute = time.perf_counter() - t0

speedup = t_brute / t_balltree if t_balltree > 0 else float("inf")

record(
    "V11 speed",
    True,
    f"BallTree {t_balltree:.3f}s, brute-force {t_brute:.3f}s, {speedup:.1f}x speedup"
)

agree = int((indices.flatten() == brute_indices).sum())
print(f"BallTree/brute-force index agreement: {agree:,}/{bench_n:,}")

[pass] V11 speed: BallTree 0.521s, brute-force 68.702s, 131.8x speedup
BallTree/brute-force index agreement: 10,000/10,000


In [19]:
# ── V12 Year gap distribution ───────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["year_gap"]).drop_nulls()

    if df.height == 0:
        record(f"V12 year_gap dist {year}", False, "no filled rows")
        continue

    g = df["year_gap"].cast(pl.Int64)

    record(
        f"V12 year_gap dist {year}",
        True,
        f"median={g.median():.0f}, p90={g.quantile(0.9):.0f}, max={g.max()}, n={df.height:,}"
    )


[pass] V12 year_gap dist 2016: median=6, p90=7, max=12, n=485,469
[pass] V12 year_gap dist 2017: median=5, p90=6, max=13, n=472,854
[pass] V12 year_gap dist 2018: median=4, p90=5, max=14, n=478,170
[pass] V12 year_gap dist 2019: median=3, p90=4, max=15, n=492,698
[pass] V12 year_gap dist 2020: median=2, p90=3, max=16, n=412,779
[pass] V12 year_gap dist 2021: median=2, p90=2, max=17, n=479,606
[pass] V12 year_gap dist 2022: median=1, p90=3, max=18, n=483,860
[pass] V12 year_gap dist 2023: median=1, p90=4, max=19, n=367,744
[pass] V12 year_gap dist 2024: median=2, p90=5, max=20, n=475,897


In [20]:
# ── V13 AADT range validation ───────────────────────────────────────────────

for year, path in year_files:
    df = pl.read_csv(path, columns=["Adt_Curnt_Amt"]).drop_nulls()

    if df.height == 0:
        record(f"V13 AADT range {year}", False, "no filled rows")
        continue

    v = df["Adt_Curnt_Amt"]

    p5 = v.quantile(0.05)
    p95 = v.quantile(0.95)
    median = v.median()

    too_low = (v < 10).sum()
    too_high = (v > 500_000).sum()

    ok = too_low < df.height * 0.01 and too_high < df.height * 0.01

    record(
        f"V13 AADT range {year}",
        ok,
        f"median={median:,.0f}, p5={p5:,.0f}, p95={p95:,.0f}, <10: {too_low}, >500k: {too_high}"
    )


[pass] V13 AADT range 2016: median=6,786, p5=220, p95=37,329, <10: 600, >500k: 0
[pass] V13 AADT range 2017: median=6,818, p5=214, p95=37,732, <10: 660, >500k: 0
[pass] V13 AADT range 2018: median=7,042, p5=221, p95=39,340, <10: 681, >500k: 0
[pass] V13 AADT range 2019: median=7,325, p5=233, p95=40,162, <10: 668, >500k: 0
[pass] V13 AADT range 2020: median=6,280, p5=192, p95=35,853, <10: 686, >500k: 0
[pass] V13 AADT range 2021: median=7,151, p5=231, p95=39,716, <10: 721, >500k: 0
[pass] V13 AADT range 2022: median=7,362, p5=240, p95=40,395, <10: 627, >500k: 0
[pass] V13 AADT range 2023: median=7,646, p5=247, p95=41,978, <10: 487, >500k: 0
[pass] V13 AADT range 2024: median=7,859, p5=250, p95=43,518, <10: 651, >500k: 0


In [21]:
# ── V14 Fill rate validation ────────────────────────────────────────────────

total_rows = 0
total_filled = 0

for year, path in year_files:
    df = pl.read_csv(path, columns=["Adt_Curnt_Amt"])

    total = df.height
    filled = df.select(pl.col("Adt_Curnt_Amt").is_not_null().sum()).item()

    pct = 100.0 * filled / total if total > 0 else 0.0

    total_rows += total
    total_filled += filled

    record(
        f"V14 fill rate {year}",
        True,
        f"{filled:,}/{total:,} ({pct:.2f}%)"
    )

overall = 100.0 * total_filled / total_rows if total_rows > 0 else 0.0
print(f"all years: {total_filled:,}/{total_rows:,} = {overall:.2f}%")


[pass] V14 fill rate 2016: 478,562/497,189 (96.25%)
[pass] V14 fill rate 2017: 466,324/484,825 (96.18%)
[pass] V14 fill rate 2018: 471,780/490,927 (96.10%)
[pass] V14 fill rate 2019: 485,867/504,885 (96.23%)
[pass] V14 fill rate 2020: 407,145/423,987 (96.03%)
[pass] V14 fill rate 2021: 472,915/491,698 (96.18%)
[pass] V14 fill rate 2022: 477,165/496,021 (96.20%)
[pass] V14 fill rate 2023: 362,885/377,107 (96.23%)
[pass] V14 fill rate 2024: 469,788/488,479 (96.17%)
all years: 4,092,431/4,255,118 = 96.18%


In [22]:
# ── Save validation results ─────────────────────────────────────────────────

summary = pl.DataFrame(results).with_columns(
    pl.when(pl.col("passed"))
    .then(pl.lit("pass"))
    .otherwise(pl.lit("FAIL"))
    .alias("status")
).select([
    "status",
    "validation",
    "passed",
    "msg"
])

n_pass = summary.select(pl.col("passed").sum()).item()
n_fail = summary.height - n_pass

print(f"{n_pass} pass, {n_fail} fail (of {summary.height})")
print(summary)

summary.write_csv(os.path.join(OUTPUTS_DIR, "validation_results.csv"))

print(f"saved {os.path.join(OUTPUTS_DIR, 'validation_results.csv')}")

110 pass, 0 fail (of 110)
shape: (110, 4)
┌────────┬────────────────────┬────────┬──────────────────────────────┐
│ status ┆ validation         ┆ passed ┆ msg                          │
│ ---    ┆ ---                ┆ ---    ┆ ---                          │
│ str    ┆ str                ┆ bool   ┆ str                          │
╞════════╪════════════════════╪════════╪══════════════════════════════╡
│ pass   ┆ V01 schema 2016    ┆ true   ┆ 10/10 present, 92 cols total │
│ pass   ┆ V01 schema 2017    ┆ true   ┆ 10/10 present, 92 cols total │
│ pass   ┆ V01 schema 2018    ┆ true   ┆ 10/10 present, 92 cols total │
│ pass   ┆ V01 schema 2019    ┆ true   ┆ 10/10 present, 92 cols total │
│ pass   ┆ V01 schema 2020    ┆ true   ┆ 10/10 present, 92 cols total │
│ …      ┆ …                  ┆ …      ┆ …                            │
│ pass   ┆ V14 fill rate 2020 ┆ true   ┆ 407,145/423,987 (96.03%)     │
│ pass   ┆ V14 fill rate 2021 ┆ true   ┆ 472,915/491,698 (96.18%)     │
│ pass   ┆ V14 fill ra